# IRP Stage 3 — Experiment Tracking
## Scenario 1: A single data scientist participating in an ML competition
**MLOps Group Project | Section 1, Group 5**  
*Maria-Irina Popa, Enzo Jerez, Roberto Cummings, Jia Yi Rachel Lee, Thomas Christian Matenco*

---

This scenario demonstrates how an individual data scientist can use MLflow to track ML experiments **locally**, with no remote server. All runs, metrics, parameters, and artifacts are stored in an `mlruns/` folder in the current directory.

### MLflow setup overview
- **Tracking server:** Not used (runs locally, no remote server)
- **Backend store:** Local filesystem (`mlruns/` folder)
- **Artifacts store:** Local filesystem (same `mlruns/` folder)

### How to use the MLflow UI
Run `mlflow ui` in your terminal from this folder, then open [http://localhost:5000](http://localhost:5000).

> **Tip:** This local setup is ideal for solo experimentation. For team access, use Scenario 2 with a tracking server.

---
## Part 1 — Setup & Data Pipeline

In [ ]:
import time
import warnings
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, f1_score,
    recall_score, precision_score, confusion_matrix
)

import mlflow
import mlflow.sklearn
import sklearn

warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

### MLflow Tracking Setup

In Scenario 1, we do **not** set a tracking URI — MLflow defaults to storing everything locally in `mlruns/`. No server needs to be running.

In [ ]:
# No tracking server — MLflow saves runs locally in ./mlruns/
print(f"Tracking URI: '{mlflow.get_tracking_uri()}'")

In [ ]:
mlflow.search_experiments()

### Config — Single Source of Truth

In [ ]:
DATA_PATH = "../data/retail_store_inventory.csv"

# Labelling thresholds
THETA_LOW   = 1.2
THETA_HIGH  = 4.5
SALES_VEL   = 0.8

# Temporal split dates
CUTOFF_VAL  = "2023-07-01"
CUTOFF_TEST = "2023-11-01"

RANDOM_STATE = 42
SAMPLE_FRAC  = 1.0   # 0.1 = fast iteration | 1.0 = full dataset

print(f"THETA_LOW={THETA_LOW} | THETA_HIGH={THETA_HIGH} | SALES_VEL={SALES_VEL}")
print(f"CUTOFF_VAL={CUTOFF_VAL} | CUTOFF_TEST={CUTOFF_TEST}")

### Load and Prepare Data

In [ ]:
t0 = time.time()
df_orig = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df_orig = df_orig.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)

df = df_orig.copy()
if SAMPLE_FRAC < 1.0:
    combos = df[["Store ID", "Product ID"]].drop_duplicates().sample(
        frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    df = df.merge(combos, on=["Store ID", "Product ID"])
    df = df.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)

# Fix datatypes
cat_cols = ["Store ID", "Product ID", "Category", "Region", "Weather Condition", "Seasonality"]
df[cat_cols] = df[cat_cols].astype("category")
df["Holiday/Promotion"] = df["Holiday/Promotion"].astype(bool)
df["Demand_Forecast_Clean"] = df["Demand Forecast"].clip(lower=1)

print(f"Loaded {len(df):,} rows in {time.time()-t0:.2f}s")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

### Inventory Reconstruction

In [ ]:
def reconstruct_inventory(group):
    group = group.sort_values("Date").copy()
    sold    = group["Units Sold"].values
    ordered = group["Units Ordered"].values
    n   = len(group)
    inv = np.empty(n, dtype=np.float64)
    inv[0] = group["Inventory Level"].iloc[0]
    for i in range(1, n):
        inv[i] = max(inv[i-1] - sold[i-1] + ordered[i-1], 0.0)
    group["Inventory_Reconstructed"] = inv
    return group

reconstructed = []
for (_, _), group in df.groupby(["Store ID", "Product ID"], sort=False):
    reconstructed.append(reconstruct_inventory(group))

df = pd.concat(reconstructed).sort_values(
    ["Store ID", "Product ID", "Date"]).reset_index(drop=True)
print("Inventory reconstruction complete ✓")

### Feature Engineering (11 Features)

In [ ]:
# Time-series features
df["Inventory_Lag1"]     = df.groupby(["Store ID", "Product ID"])["Inventory_Reconstructed"].shift(1)
df["Units_Sold_Lag1"]    = df.groupby(["Store ID", "Product ID"])["Units Sold"].shift(1)
df["Rolling7_Inventory"] = (
    df.groupby(["Store ID", "Product ID"])["Inventory_Reconstructed"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# Depletion features
df["Inventory_Change"]     = df["Inventory_Reconstructed"] - df["Inventory_Lag1"]
df["Inventory_Change_Pct"] = (
    df["Inventory_Change"] / df["Inventory_Lag1"].replace(0, np.nan)
).fillna(0)
df["Days_of_Stock"] = (
    df["Inventory_Reconstructed"] / df["Units Sold"].replace(0, np.nan)
).fillna(df["Inventory_Reconstructed"] / df["Demand_Forecast_Clean"])
df["Days_of_Stock"] = df["Days_of_Stock"].replace([np.inf, -np.inf], np.nan)
df["Inventory_vs_Rolling7"] = df["Inventory_Reconstructed"] - df["Rolling7_Inventory"]

# Ratio features
df["Sales_Velocity"] = (
    df["Units Sold"] / df["Inventory_Reconstructed"].replace(0, np.nan)
).fillna(0).replace([np.inf, -np.inf], np.nan)
df["Coverage_Ratio"]     = df["Inventory_Reconstructed"] / df["Demand_Forecast_Clean"]
df["Forecast_Error"]     = df["Units Sold"] - df["Demand Forecast"]
df["Order_to_Inventory"] = (
    df["Units Ordered"] / df["Inventory_Reconstructed"].replace(0, np.nan)
).fillna(0).replace([np.inf, -np.inf], 0)

# Drop warm-up rows (first 7 days per series)
df = df.dropna(subset=["Inventory_Lag1", "Rolling7_Inventory"]).reset_index(drop=True)
print(f"Feature engineering complete ✓  ({len(df):,} rows remaining)")

### Labelling (t+1 shift)

In [ ]:
df["Risk_Label_Current"] = "Safe Zone"
df.loc[df["Inventory Level"] < df["Demand_Forecast_Clean"] * THETA_LOW,
       "Risk_Label_Current"] = "Stockout Risk"
df.loc[(df["Inventory Level"] > df["Demand_Forecast_Clean"] * THETA_HIGH) &
       (df["Units Sold"] < df["Demand_Forecast_Clean"] * SALES_VEL),
       "Risk_Label_Current"] = "Overstock Risk"

df["Risk_Label"] = df.groupby(["Store ID", "Product ID"])["Risk_Label_Current"].shift(-1)
df = df.dropna(subset=["Risk_Label"]).reset_index(drop=True)

print("Class distribution:")
counts = df["Risk_Label"].value_counts()
for label, count in counts.items():
    print(f"  {label:<18} {count:>6,}  ({count/len(df)*100:.1f}%)")

### Temporal Split & Feature Encoding

In [ ]:
train = df[df["Date"] <  pd.Timestamp(CUTOFF_VAL)].copy()
val   = df[(df["Date"] >= pd.Timestamp(CUTOFF_VAL)) & (df["Date"] < pd.Timestamp(CUTOFF_TEST))].copy()
test  = df[df["Date"] >= pd.Timestamp(CUTOFF_TEST)].copy()

# Impute NaN using train-only medians
for col in ["Days_of_Stock", "Sales_Velocity"]:
    train_median = train[col].median()
    train[col] = train[col].fillna(train_median)
    val[col]   = val[col].fillna(train_median)
    test[col]  = test[col].fillna(train_median)

FEATURES = [
    "Inventory Level", "Units Sold", "Demand Forecast",
    "Price", "Discount", "Competitor Pricing", "Holiday/Promotion",
    "Inventory_Change", "Inventory_Change_Pct", "Days_of_Stock",
    "Inventory_vs_Rolling7", "Sales_Velocity",
    "Inventory_Lag1", "Units_Sold_Lag1", "Rolling7_Inventory",
    "Coverage_Ratio", "Forecast_Error", "Order_to_Inventory",
]
CATEGORICAL_COLS = ["Category", "Region", "Weather Condition", "Seasonality"]

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    le.fit(train[col])
    enc_col = col + "_enc"
    train[enc_col] = le.transform(train[col])
    val[enc_col]   = le.transform(val[col])
    test[enc_col]  = le.transform(test[col])
    FEATURES.append(enc_col)

X_train, y_train = train[FEATURES], train["Risk_Label"]
X_val,   y_val   = val[FEATURES],   val["Risk_Label"]
X_test,  y_test  = test[FEATURES],  test["Risk_Label"]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
assert train["Date"].max() < pd.Timestamp(CUTOFF_VAL)
print("✅ No temporal leakage confirmed")

---
## Part 2 — MLflow Experiment Tracking

Each model is logged as a separate run in the same experiment. Parameters, metrics, confusion matrix artifacts, and the model itself are all tracked locally in `mlruns/`.

In [ ]:
mlflow.set_experiment("irp-inventory-risk-prediction")

In [ ]:
def log_run(model, model_name, params, X_tr, y_tr, X_v, y_v):
    """Train a model and log everything to MLflow."""
    labels_order = ["Stockout Risk", "Overstock Risk", "Safe Zone"]

    with mlflow.start_run(run_name=model_name) as run:

        # --- Parameters ---
        mlflow.log_param("model", model_name)
        mlflow.log_params(params)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        mlflow.log_param("theta_low", THETA_LOW)
        mlflow.log_param("theta_high", THETA_HIGH)
        mlflow.log_param("sales_vel", SALES_VEL)
        mlflow.log_param("n_features", len(FEATURES))
        mlflow.log_param("train_size", len(X_tr))
        mlflow.log_param("val_size", len(X_v))

        # --- Train ---
        t0 = time.time()
        model.fit(X_tr, y_tr)
        train_time = time.time() - t0

        # --- Metrics ---
        preds_val = model.predict(X_v)
        f1_val        = f1_score(y_v, preds_val, average="weighted")
        stockout_rec  = recall_score(y_v, preds_val, labels=["Stockout Risk"],  average="macro")
        overstock_rec = recall_score(y_v, preds_val, labels=["Overstock Risk"], average="macro")
        stockout_prec = precision_score(y_v, preds_val, labels=["Stockout Risk"],  average="macro")

        mlflow.log_metric("val_weighted_f1",   f1_val)
        mlflow.log_metric("val_stockout_recall", stockout_rec)
        mlflow.log_metric("val_overstock_recall", overstock_rec)
        mlflow.log_metric("val_stockout_precision", stockout_prec)
        mlflow.log_metric("train_time_sec", train_time)

        # --- Artifact: Confusion matrix heatmap ---
        cm = confusion_matrix(y_v, preds_val, labels=labels_order)
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        annot = np.empty_like(cm, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                annot[i, j] = f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)"

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=annot, fmt="", cmap="Blues",
                    xticklabels=labels_order, yticklabels=labels_order,
                    linewidths=0.5, ax=ax)
        ax.set_title(f"Confusion Matrix — {model_name} (Validation Set)", fontweight="bold")
        ax.set_ylabel("True label")
        ax.set_xlabel("Predicted label")
        fig.tight_layout()
        fig.savefig("confusion_matrix.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix.png")

        # --- Log model ---
        mlflow.sklearn.log_model(model, name="model",
                                  input_example=X_tr.iloc[:1])

        # --- Tags ---
        mlflow.set_tag("model_type", type(model).__name__)
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("stage", "stage3")
        mlflow.set_tag("description", f"{model_name} on IRP dataset — local tracking")

        print(f"✓ {model_name:<35}  val F1={f1_val:.4f}  "
              f"stockout_recall={stockout_rec:.4f}  "
              f"train_time={train_time:.1f}s  "
              f"run_id={run.info.run_id[:8]}...")
    return run.info.run_id

### Run 1 — Logistic Regression Baseline (from Stage 2)

In [ ]:
lr_params = {"max_iter": 1000, "class_weight": "balanced", "random_state": RANDOM_STATE}
lr = LogisticRegression(**lr_params)

run_id_lr = log_run(lr, "LogisticRegression", lr_params, X_train, y_train, X_val, y_val)

### Run 2 — Random Forest (default)

In [ ]:
rf_params = {"n_estimators": 100, "class_weight": "balanced", "random_state": RANDOM_STATE, "n_jobs": -1}
rf = RandomForestClassifier(**rf_params)

run_id_rf = log_run(rf, "RandomForest_100", rf_params, X_train, y_train, X_val, y_val)

### Run 3 — Random Forest (deeper)

In [ ]:
rf2_params = {"n_estimators": 200, "max_depth": 15, "class_weight": "balanced",
              "random_state": RANDOM_STATE, "n_jobs": -1}
rf2 = RandomForestClassifier(**rf2_params)

run_id_rf2 = log_run(rf2, "RandomForest_200_d15", rf2_params, X_train, y_train, X_val, y_val)

In [ ]:
# Confirm all runs saved locally
mlflow.search_experiments()

---
## Part 3 — Compare Runs & Evaluate Best Model on Test Set

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = client.get_experiment_by_name("irp-inventory-risk-prediction")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.val_weighted_f1 DESC"]
)

print(f"{'Run Name':<35} {'Val F1':>8} {'Stockout Rec':>14}")
print("-" * 60)
for r in runs:
    name = r.data.params.get("model", "?")
    f1   = r.data.metrics.get("val_weighted_f1", float("nan"))
    rec  = r.data.metrics.get("val_stockout_recall", float("nan"))
    print(f"{name:<35} {f1:>8.4f} {rec:>14.4f}")

In [ ]:
# Load best model and evaluate on test set
best_run = runs[0]
best_run_id = best_run.info.run_id
best_model_name = best_run.data.params.get("model", "unknown")

best_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

preds_test = best_model.predict(X_test)
f1_test = f1_score(y_test, preds_test, average="weighted")

print(f"Best model   : {best_model_name}")
print(f"Test Weighted F1 : {f1_test:.4f}")
print()
print(classification_report(y_test, preds_test))

### Launch MLflow UI

In [ ]:
import subprocess
subprocess.Popen(["mlflow", "ui"])
print("MLflow UI started. Open http://localhost:5000 in your browser.")